In [ ]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
===========================================================
VIN Odometer Loss Summary Tool (Tkinter GUI)
===========================================================

Author   : Sanmathi S
Year     : 2026
Version  : 1.0
Purpose  :
    This tool provides a Tkinter-based GUI to process multiple CSV files
    containing vehicle odometer readings. It calculates odometer loss
    information per VIN and generates a summary table.

Workflow :
    1. Select a folder containing CSV files.
    2. For each file:
        - Validate required columns: eventTime, odometer, Vin.
        - Sort rows by eventTime.
        - Calculate odometer loss when differences > 0.125 km.
        - Aggregate total odometer loss per VIN.
    3. Display results in a styled Treeview table with zebra striping.
    4. Save summary to CSV or clear the table via GUI buttons.

Output   :
    - Interactive GUI table showing VIN and overall odometer loss.
    - Option to save summary as a CSV file.
    - Warning messages if no valid data is found.

Dependencies:
    - pandas
    - os
    - tkinter
    - ttk (Treeview styling)

Usage    :
    python vin_odometer_loss_summary.py

Notes    :
    - Ensure input CSV files contain columns: eventTime, odometer, Vin.
    - Odometer loss is flagged when differences exceed 0.125 km.
===========================================================
"""


import pandas as pd
import os
import tkinter as tk
from tkinter import filedialog, messagebox, ttk

def process_folder(folder_path):
    summary_rows = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            try:
                df = pd.read_csv(file_path)
                if not {"eventTime", "odometer", "Vin"}.issubset(df.columns):
                    continue

                df.sort_values(by="eventTime", inplace=True)
                df["odometer_loss_info"] = 0.0

                for i in range(1, len(df)):
                    current = df.loc[i, "odometer"]
                    previous = df.loc[i - 1, "odometer"]
                    if current != 0 and previous != 0:
                        diff = current - previous
                        if diff > 0.125:
                            df.loc[i, "odometer_loss_info"] = diff

                total_loss = df["odometer_loss_info"].sum()
                vin_value = df.loc[0, "Vin"]
                summary_rows.append({"Vin": vin_value, "overall_Odoloss (Km)": round(total_loss, 2)})

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    return pd.DataFrame(summary_rows)

def browse_folder():
    folder_path = filedialog.askdirectory()
    if folder_path:
        summary_df = process_folder(folder_path)
        if summary_df.empty:
            messagebox.showwarning("No Data", "No valid CSV files found.")
            return

        for row in tree.get_children():
            tree.delete(row)

        for i, row in summary_df.iterrows():
            tag = "evenrow" if i % 2 == 0 else "oddrow"
            tree.insert("", "end", values=(row["Vin"], row["overall_Odoloss (Km)"]), tags=(tag,))

        save_button.config(state="normal")
        clear_button.config(state="normal")
        app.summary_df = summary_df

def save_summary():
    file_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV files", "*.csv")])
    if file_path:
        app.summary_df.to_csv(file_path, index=False)
        messagebox.showinfo("Saved", f"Summary saved to:\n{file_path}")

def clear_table():
    for row in tree.get_children():
        tree.delete(row)
    save_button.config(state="disabled")
    clear_button.config(state="disabled")
    app.summary_df = pd.DataFrame()

# GUI Setup
app = tk.Tk()
app.title("VIN Odometer Loss Summary")
app.geometry("700x500")
app.configure(bg="#ecf0f1")
app.summary_df = pd.DataFrame()

# Header
# fg="#34495e"
# font=("Segoe UI", 18, "bold")
# font=('Arial Black', 15, 'bold')
header = tk.Label(app, text="📊 VIN Odometer Loss Summary", font=("Arial Black", 15, "bold"), bg="#ecf0f1", fg="#353E9E")
header.pack(pady=15)

# Button Frame
button_frame = tk.Frame(app, bg="#ecf0f1")
button_frame.pack(pady=10)

browse_button = tk.Button(button_frame, text="📁  Select Folder", command=browse_folder, font=("Segoe UI", 12), bg="#2980b9", fg="white", padx=12, pady=6)
browse_button.pack(side="left", padx=10)

save_button = tk.Button(button_frame, text="💾  Save Summary", command=save_summary, font=("Segoe UI", 12), bg="#27ae60", fg="white", padx=12, pady=6, state="disabled")
save_button.pack(side="left", padx=10)

clear_button = tk.Button(button_frame, text="🧹  Clear Table", command=clear_table, font=("Segoe UI", 12), bg="#c0392b", fg="white", padx=12, pady=6, state="disabled")
clear_button.pack(side="left", padx=10)

# Table Styling
style = ttk.Style()
style.theme_use("default")
style.configure("Treeview",
    background="#ffffff",
    foreground="#2c3e50",
    rowheight=30,
    fieldbackground="#ffffff",
    font=("Segoe UI", 11)
)
style.configure("Treeview.Heading",
    font=("Arial Black", 11, "bold"),
    background="#787ae0",
    foreground="#FFFFFF"
)
style.map("Treeview", background=[("selected", "#3498db")])

tree = ttk.Treeview(app, columns=("Vin", "overall_Odoloss (Km)"), show="headings")
tree.heading("Vin", text="VIN")
tree.heading("overall_Odoloss (Km)", text="Odometer Loss")
tree.column("Vin", width=350, anchor="center")
tree.column("overall_Odoloss (Km)", width=200, anchor="center")


def on_hover(event):
    region = tree.identify_region(event.x, event.y)
    if region == "heading":
        style.configure("Treeview.Heading", foreground="black")
    else:
        style.configure("Treeview.Heading", foreground="#FFFFFF")

tree.bind("<Motion>", on_hover)

tree.tag_configure("evenrow", background="#cad3eb")
tree.tag_configure("oddrow", background="#cbf0e8")



tree.pack(expand=True, fill="both", padx=20, pady=10)

app.mainloop()